In [1]:
import duckdb

In [2]:
con = duckdb.connect(database=':memory:')

In [18]:
con.execute("""
CREATE TABLE IF NOT EXISTS stop_times AS
            SELECT * from read_csv_auto('*stop_times_combined.csv',
            header = True,
            AUTO_DETECT = True)
""")
con.execute("""
CREATE TABLE IF NOT EXISTS trips AS
            SELECT * from read_csv_auto('*trips_combined.csv',
            header = True,
            AUTO_DETECT = True)
""")

In [5]:
con.execute("SELECT * from stop_times LIMIT 5").df()

,trip_uid,stop_id,track,arrival_time,departure_time,last_observed,marked_past
0,1735707600_7..S,701S,2,<NA>,1735725600,1735725601,1735725605
1,1735707600_7..S,702S,1,1735725730,1735725750,1735725755,1735725766
2,1735707600_7..S,705S,1,1735725837,1735725857,1735725860,1735725871
3,1735707600_7..S,706S,1,1735725916,1735725936,1735725980,1735725994
4,1735707600_7..S,707S,1,1735726133,1735726153,1735726085,1735726099


In [19]:
con.execute("SELECT * from trips LIMIT 5").df()

,trip_uid,trip_id,route_id,direction_id,start_time,vehicle_id,last_observed,marked_past,num_updates,num_schedule_changes,num_schedule_rewrites
0,1735707600_7..S,030000_7..S,7,1,1735707600,07 0500 MST/34H,1735727990,1735728004,575,0,0
1,1735707600_GS.N01R,030000_GS.N01R,GS,0,1735707600,0S 0500 GCS/TSS,1735725680,1735725691,282,0,0
2,1735707720_5..S32R,030200_5..S32R,5,1,1735707720,05 0502 DYR/180,1735726415,1735726429,379,0,0
3,1735707750_2..S08R,030250_2..S08R,2,1,1735707750,02 0502+ 241/FLA,1735732086,1735732100,1090,0,0
4,1735707750_6..N01R,030250_6..N01R,6,0,1735707750,06 0502+ BBR/PEL,1735724300,1735725751,16,0,0


In [52]:
con.execute("""
SELECT trip_uid, 
            stop_id, 
            track, 
            to_timestamp(arrival_time) as arrival, 
            to_timestamp(arrival_time) - INTERVAL (minute(to_timestamp(arrival_time)) % 10) MINUTE
            - INTERVAL (second(to_timestamp(arrival_time)) % 60) SECOND as arrival_interval, 
            to_timestamp(departure_time) as departure,
            to_timestamp(departure_time) - INTERVAL (minute(to_timestamp(departure_time)) % 10) MINUTE
            - INTERVAL (second(to_timestamp(departure_time)) % 60) SECOND as departure_interval, 
            to_timestamp(last_observed) as last_seen,
            to_timestamp(last_observed) - INTERVAL (minute(to_timestamp(last_observed)) % 10) MINUTE
            - INTERVAL (second(to_timestamp(last_observed)) % 60) SECOND as last_seen_interval, 
            to_timestamp(marked_past)as marked_past_time,
            to_timestamp(marked_past) - INTERVAL (minute(to_timestamp(marked_past)) % 10) MINUTE
            - INTERVAL (second(to_timestamp(marked_past)) % 60) SECOND as marked_past_interval, 
FROM
            stop_times
WHERE
track IS NOT NULL 
            AND
            arrival IS NOT NULL
            AND
            departure IS NOT NULL
            AND
            marked_past_time IS NOT NULL
""").df().head(5)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,trip_uid,stop_id,track,arrival,arrival_interval,departure,departure_interval,last_seen,last_seen_interval,marked_past_time,marked_past_interval
0,1735707600_7..S,702S,1,2025-01-01 05:02:10-05:00,2025-01-01 05:00:00-05:00,2025-01-01 05:02:30-05:00,2025-01-01 05:00:00-05:00,2025-01-01 05:02:35-05:00,2025-01-01 05:00:00-05:00,2025-01-01 05:02:46-05:00,2025-01-01 05:00:00-05:00
1,1735707600_7..S,705S,1,2025-01-01 05:03:57-05:00,2025-01-01 05:00:00-05:00,2025-01-01 05:04:17-05:00,2025-01-01 05:00:00-05:00,2025-01-01 05:04:20-05:00,2025-01-01 05:00:00-05:00,2025-01-01 05:04:31-05:00,2025-01-01 05:00:00-05:00
2,1735707600_7..S,706S,1,2025-01-01 05:05:16-05:00,2025-01-01 05:00:00-05:00,2025-01-01 05:05:36-05:00,2025-01-01 05:00:00-05:00,2025-01-01 05:06:20-05:00,2025-01-01 05:00:00-05:00,2025-01-01 05:06:34-05:00,2025-01-01 05:00:00-05:00
3,1735707600_7..S,707S,1,2025-01-01 05:08:53-05:00,2025-01-01 05:00:00-05:00,2025-01-01 05:09:13-05:00,2025-01-01 05:00:00-05:00,2025-01-01 05:08:05-05:00,2025-01-01 05:00:00-05:00,2025-01-01 05:08:19-05:00,2025-01-01 05:00:00-05:00
4,1735707600_7..S,708S,1,2025-01-01 05:10:30-05:00,2025-01-01 05:10:00-05:00,2025-01-01 05:10:50-05:00,2025-01-01 05:10:00-05:00,2025-01-01 05:09:35-05:00,2025-01-01 05:00:00-05:00,2025-01-01 05:09:49-05:00,2025-01-01 05:00:00-05:00


In [ ]:
con.execute("""
SELECT 
            trip_uid,
            trip_id,
            route_id,
            direction_id,
            to_timestamp(start_time) as start,
            vehicle_id,
            to_timestamp(last_observed) as last_seen,
            to_timestamp(marked_past) as marked_past_time,
            num_updates,
            num_schedule_changes,
            num_schedule_rewrites
FROM trips
            WHERE marked_past_time IS NOT NULL
""").df()[100:120]

,trip_uid,trip_id,route_id,direction_id,start,vehicle_id,last_seen,marked_past_time,num_updates,num_schedule_changes,num_schedule_rewrites
0,1735707600_7..S,030000_7..S,7,1,2025-01-01 00:00:00-05:00,07 0500 MST/34H,2025-01-01 05:39:50-05:00,2025-01-01 05:40:04-05:00,575,0,0
1,1735707600_GS.N01R,030000_GS.N01R,GS,0,2025-01-01 00:00:00-05:00,0S 0500 GCS/TSS,2025-01-01 05:01:20-05:00,2025-01-01 05:01:31-05:00,282,0,0
2,1735707720_5..S32R,030200_5..S32R,5,1,2025-01-01 00:02:00-05:00,05 0502 DYR/180,2025-01-01 05:13:35-05:00,2025-01-01 05:13:49-05:00,379,0,0
3,1735707750_2..S08R,030250_2..S08R,2,1,2025-01-01 00:02:30-05:00,02 0502+ 241/FLA,2025-01-01 06:48:06-05:00,2025-01-01 06:48:20-05:00,1090,0,0
4,1735707750_6..N01R,030250_6..N01R,6,0,2025-01-01 00:02:30-05:00,06 0502+ BBR/PEL,2025-01-01 04:38:20-05:00,2025-01-01 05:02:31-05:00,16,0,0


In [50]:
con.execute("""
SELECT 
            trip_uid,
            trip_id,
            route_id,
            direction_id,
            to_timestamp(start_time) as start,
            to_timestamp(start_time) - INTERVAL (minute(to_timestamp(start_time)) % 10) MINUTE
            - INTERVAL (second(to_timestamp(start_time)) % 60) SECOND as start_interval, 
            vehicle_id,
            to_timestamp(last_observed) as last_seen,
            to_timestamp(last_observed) - INTERVAL (minute(to_timestamp(last_observed)) % 10) MINUTE
            - INTERVAL (second(to_timestamp(last_observed)) % 60) SECOND as last_seen_interval, 
            to_timestamp(marked_past) as marked_past_time,
            to_timestamp(marked_past) - INTERVAL (minute(to_timestamp(marked_past)) % 10) MINUTE
            - INTERVAL (second(to_timestamp(marked_past)) % 60) SECOND as marked_past_interval, 
            num_updates,
            num_schedule_changes,
            num_schedule_rewrites
FROM trips
            WHERE
            marked_past_time IS NOT NULL
            AND
            marked_past_interval IS NOT NULL
""").df()[100:120]

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,trip_uid,trip_id,route_id,direction_id,start,start_interval,vehicle_id,last_seen,last_seen_interval,marked_past_time,marked_past_interval,num_updates,num_schedule_changes,num_schedule_rewrites
100,1735713600_4..N01R,040000_4..N01R,4,0,2025-01-01 01:40:00-05:00,2025-01-01 01:40:00-05:00,04 0640 NLT/WDL,2025-01-01 08:00:35-05:00,2025-01-01 08:00:00-05:00,2025-01-01 08:00:39-05:00,2025-01-01 08:00:00-05:00,916,0,0
101,1735713600_5..N01R,040000_5..N01R,5,0,2025-01-01 01:40:00-05:00,2025-01-01 01:40:00-05:00,05 0640 180/DYR,2025-01-01 06:52:20-05:00,2025-01-01 06:50:00-05:00,2025-01-01 06:52:21-05:00,2025-01-01 06:50:00-05:00,369,0,0
102,1735713600_GS.N01R,040000_GS.N01R,GS,0,2025-01-01 01:40:00-05:00,2025-01-01 01:40:00-05:00,0S 0640 GCS/TSS,2025-01-01 06:41:51-05:00,2025-01-01 06:40:00-05:00,2025-01-01 06:42:05-05:00,2025-01-01 06:40:00-05:00,286,0,0
103,1735713660_7..S,040100_7..S,7,1,2025-01-01 01:41:00-05:00,2025-01-01 01:40:00-05:00,07 0641 MST/34H,2025-01-01 07:18:05-05:00,2025-01-01 07:10:00-05:00,2025-01-01 07:18:09-05:00,2025-01-01 07:10:00-05:00,567,0,0
104,1735713750_1..S03R,040250_1..S03R,1,1,2025-01-01 01:42:30-05:00,2025-01-01 01:40:00-05:00,01 0642+ 242/SFT,2025-01-01 07:39:35-05:00,2025-01-01 07:30:00-05:00,2025-01-01 07:39:39-05:00,2025-01-01 07:30:00-05:00,743,0,0
105,1735713840_3..S42R,040400_3..S42R,3,1,2025-01-01 01:44:00-05:00,2025-01-01 01:40:00-05:00,03 0644 148/TSQ,2025-01-01 07:01:51-05:00,2025-01-01 07:00:00-05:00,2025-01-01 07:02:05-05:00,2025-01-01 07:00:00-05:00,406,0,0
106,1735713840_GS.S01R,040400_GS.S01R,GS,1,2025-01-01 01:44:00-05:00,2025-01-01 01:40:00-05:00,0S 0644 TSS/GCS,2025-01-01 06:45:21-05:00,2025-01-01 06:40:00-05:00,2025-01-01 06:45:35-05:00,2025-01-01 06:40:00-05:00,274,0,0
107,1735713990_2..N08R,040650_2..N08R,2,0,2025-01-01 01:46:30-05:00,2025-01-01 01:40:00-05:00,02 0646+ FLA/241,2025-01-01 08:24:42-05:00,2025-01-01 08:20:00-05:00,2025-01-01 08:24:50-05:00,2025-01-01 08:20:00-05:00,1069,0,0
108,1735714020_3..N42R,040700_3..N42R,3,0,2025-01-01 01:47:00-05:00,2025-01-01 01:40:00-05:00,03 0647 TSQ/148,2025-01-01 07:10:24-05:00,2025-01-01 07:10:00-05:00,2025-01-01 07:10:35-05:00,2025-01-01 07:10:00-05:00,474,0,0
109,1735714080_6..N01R,040800_6..N01R,6,0,2025-01-01 01:48:00-05:00,2025-01-01 01:40:00-05:00,06 0648 BBR/PEL,2025-01-01 07:45:39-05:00,2025-01-01 07:40:00-05:00,2025-01-01 07:45:50-05:00,2025-01-01 07:40:00-05:00,716,0,0


In [25]:
con.execute("SUMMARIZE stop_times").df()

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,trip_uid,VARCHAR,1735707600_7..S,1760155200_L..N,1748646,None,None,None,None,None,57632042,0.00
1,stop_id,VARCHAR,101N,X22N,1143,None,None,None,None,None,57632042,0.00
2,track,VARCHAR,-P,Y2,116,None,None,None,None,None,57632042,0.04
3,arrival_time,BIGINT,1735704070,1760176120,16435883,1748039309.9843125,7071963.527485245,1741938677,1748090317,1754148199,57632042,1.70
4,departure_time,BIGINT,1735704000,1760176120,17691158,1748040384.1336658,7071960.76878239,1741910086,1748123796,1754167967,57632042,1.81
5,last_observed,BIGINT,1735703949,1760173198,11893144,1748041101.9756308,7072095.844518129,1741918189,1748107196,1754145850,57632042,0.00
6,marked_past,BIGINT,1735703955,1760173189,12318744,1748064593.1924293,7065291.221091474,1741962090,1748142868,1754185319,57632042,0.38


In [26]:
con.execute("SUMMARIZE trips").df()

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,trip_uid,VARCHAR,1735707600_7..S,1760155200_L..N,1748646,None,None,None,None,None,2143572,0.00
1,trip_id,VARCHAR,000000_1..N03R,287950_3..N43R,191813,None,None,None,None,None,2143572,0.00
2,route_id,VARCHAR,1,Z,27,None,None,None,None,None,2143572,0.00
3,direction_id,BIGINT,0,1,2,0.5039345540994191,0.49998463566856793,0,1,1,2143572,0.00
4,start_time,BIGINT,1735707600,1760155200,599896,1747992146.119996,7078731.896722165,1741866433,1748035993,1754114838,2143572,0.00
5,vehicle_id,VARCHAR,$1 0012+ 242/SFT,W6 0032 WSQ/59S,161094,None,None,None,None,None,2143572,0.00
6,last_observed,BIGINT,1735704090,1760173198,1567104,1748010272.9274724,7078007.102361698,1741886951,1748062901,1754148744,2143572,0.00
7,marked_past,BIGINT,1735704099,1760173159,2053804,1748038044.855887,7069804.881452313,1741954876,1748090820,1754224117,2143572,0.72
8,num_updates,BIGINT,1,39382,5990,1009.2489559482957,817.6540075319938,531,803,1339,2143572,0.00
9,num_schedule_changes,BIGINT,-1,20924,126,0.07813500083038964,18.574685952555015,0,0,0,2143572,0.00


In [53]:
con.execute("""
            DROP TABLE IF EXISTS updated_stop_times;
            CREATE TABLE updated_stop_times AS
SELECT trip_uid, 
            stop_id, 
            track, 
            to_timestamp(arrival_time) as arrival, 
            to_timestamp(arrival_time) - INTERVAL (minute(to_timestamp(arrival_time)) % 10) MINUTE
            - INTERVAL (second(to_timestamp(arrival_time)) % 60) SECOND as arrival_interval, 
            to_timestamp(departure_time) as departure,
            to_timestamp(departure_time) - INTERVAL (minute(to_timestamp(departure_time)) % 10) MINUTE
            - INTERVAL (second(to_timestamp(departure_time)) % 60) SECOND as departure_interval, 
            to_timestamp(last_observed) as last_seen,
            to_timestamp(last_observed) - INTERVAL (minute(to_timestamp(last_observed)) % 10) MINUTE
            - INTERVAL (second(to_timestamp(last_observed)) % 60) SECOND as last_seen_interval, 
            to_timestamp(marked_past)as marked_past_time,
            to_timestamp(marked_past) - INTERVAL (minute(to_timestamp(marked_past)) % 10) MINUTE
            - INTERVAL (second(to_timestamp(marked_past)) % 60) SECOND as marked_past_interval, 
FROM
            stop_times
WHERE
track IS NOT NULL 
            AND
            arrival IS NOT NULL
            AND
            departure IS NOT NULL
            AND
            marked_past_time IS NOT NULL
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
con.execute("""
            DROP TABLE IF EXISTS updated_trips;
            CREATE TABLE updated_trips as
SELECT 
            trip_uid,
            trip_id,
            route_id,
            direction_id,
            to_timestamp(start_time) as start,
            to_timestamp(start_time) - INTERVAL (minute(to_timestamp(start_time)) % 10) MINUTE
            - INTERVAL (second(to_timestamp(start_time)) % 60) SECOND as start_interval, 
            vehicle_id,
            to_timestamp(last_observed) as last_seen,
            to_timestamp(last_observed) - INTERVAL (minute(to_timestamp(last_observed)) % 10) MINUTE
            - INTERVAL (second(to_timestamp(last_observed)) % 60) SECOND as last_seen_interval, 
            to_timestamp(marked_past) as marked_past_time,
            to_timestamp(marked_past) - INTERVAL (minute(to_timestamp(marked_past)) % 10) MINUTE
            - INTERVAL (second(to_timestamp(marked_past)) % 60) SECOND as marked_past_interval, 
            num_updates,
            num_schedule_changes,
            num_schedule_rewrites
FROM trips
            WHERE
            marked_past_time IS NOT NULL
            AND
            marked_past_interval IS NOT NULL
""")

In [ ]:
con.execute("""EXPORT DATABASE './database/' (FORMAT parquet)
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [55]:
con.close()